In [ ]:
# ============================================================
# Retail Intelligence Brasil
# Notebook.......: 04_bronze_regioes
# Camada.........: Bronze
# Fonte..........: IBGE - API Localidades
# Objetivo.......: Persistir as regioes do IBGE na camada Bronze.
# Autor..........: Cristhina Freire
# Data...........: 12/06/2026
# ============================================================



In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import requests
import json

from datetime import datetime



In [ ]:
# ============================================================
# 2. CONFIGURAÇÃO
# ============================================================

CATALOG = "retail_intelligence"
SCHEMA = "bronze"

TABLE_NAME = "ibge_regioes_raw"

FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

BASE_URL = "https://servicodados.ibge.gov.br/api/v1/localidades/regioes"

TIMEOUT = 60

CURRENT_DATE = datetime.now().strftime("%Y-%m-%d")
CURRENT_TIME = datetime.now().strftime("%Y%m%d_%H%M%S")

VOLUME_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/landing/"
    f"ibge/regioes/{CURRENT_DATE}"
)

FILE_NAME = f"regioes_{CURRENT_TIME}.json"

FILE_PATH = f"{VOLUME_PATH}/{FILE_NAME}"

print("=" * 60)
print("BRONZE - IBGE REGIÕES")
print("=" * 60)



In [ ]:
# ============================================================
# 3. EXTRAÇÃO
# ============================================================

print("Consumindo API...")

response = requests.get(
    BASE_URL,
    timeout=TIMEOUT
)

response.raise_for_status()

landing_data = response.json()

print(f"Regiões encontrados: {len(landing_data)}")



In [ ]:
# ============================================================
# 4. PERSISTÊNCIA LANDING
# ============================================================

dbutils.fs.mkdirs(VOLUME_PATH)

dbutils.fs.put(
    FILE_PATH,
    json.dumps(
        landing_data,
        ensure_ascii=False,
        indent=2
    ),
    overwrite=True
)

print(f"JSON salvo em:\n{FILE_PATH}")



In [ ]:
# ============================================================
# 5. LEITURA LANDING
# ============================================================

print("Lendo JSON salvo no Volume...")

landing_df = (
    spark.read
         .option("multiline", "true")
         .json(FILE_PATH)
)

print("Leitura concluída.")



In [ ]:
# ============================================================
# 6. PERSISTÊNCIA NA BRONZE
# ============================================================

(
    landing_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(FULL_TABLE_NAME)
)

print("Tabela Bronze criada.")



In [ ]:
# ============================================================
# 7. INSPEÇÃO
# ============================================================

bronze_df = spark.table(FULL_TABLE_NAME)

print(f"Total de registros: {bronze_df.count()}")

bronze_df.printSchema()

display(bronze_df)



In [ ]:
# ============================================================
# 8. ENCERRAMENTO
# ============================================================

print("=" * 60)
print("Pipeline finalizado com sucesso.")
print("=" * 60)